# Walk-Forward Backtesting (Time Series)

Industrial baseline for forecasting evaluation:
- synthetic series (trend + multi-seasonality + noise)
- walk-forward backtesting
- multiple baselines: naive, seasonal naive, ARIMA
- metrics: MAE/RMSE

Notebook is CPU-friendly and stores outputs on execution.

In [1]:
import numpy as np
import pandas as pd
from math import sqrt

SEED = 1337
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 50)

def make_series(n=1200, season=24, noise=0.8):
    t = np.arange(n)
    trend = 0.004 * t
    s1 = 1.5 * np.sin(2*np.pi*t/season)
    s2 = 0.6 * np.sin(2*np.pi*t/(season*7))
    y = 50 + trend + s1 + s2 + rng.normal(0, noise, size=n)
    return y

y = make_series()
y[:5], len(y)

(array([50.03061458, 50.79381243, 50.69264132, 50.0283713 , 53.42055841]),
 1200)

## Metrics

In [2]:
def mae(a, b):
    a = np.asarray(a); b=np.asarray(b)
    return float(np.mean(np.abs(a-b)))

def rmse(a, b):
    a = np.asarray(a); b=np.asarray(b)
    return float(sqrt(np.mean((a-b)**2)))

## Walk-forward backtesting

In [3]:
from statsmodels.tsa.arima.model import ARIMA

h = 24  # horizon
initial = 400
step = 24
season = 24

rows = []
for start in range(initial, len(y) - h, step):
    train = y[:start]
    true = y[start:start+h]

    # naive
    pred_naive = np.repeat(train[-1], h)

    # seasonal naive
    pred_snaive = y[start-season:start-season+h]

    # ARIMA (small order)
    fit = ARIMA(train, order=(2,1,2)).fit()
    pred_arima = fit.forecast(steps=h)

    rows.append({
        'cut': start,
        'naive_mae': mae(true, pred_naive),
        'naive_rmse': rmse(true, pred_naive),
        'snaive_mae': mae(true, pred_snaive),
        'snaive_rmse': rmse(true, pred_snaive),
        'arima_mae': mae(true, pred_arima),
        'arima_rmse': rmse(true, pred_arima),
    })

res = pd.DataFrame(rows)
res.head(), res.shape

(   cut  naive_mae  naive_rmse  snaive_mae  snaive_rmse  arima_mae  arima_rmse
 0  400   1.285158    1.668477    0.744140     0.955476   1.283426    1.661271
 1  424   1.084017    1.346646    0.943326     1.197167   1.188260    1.419464
 2  448   2.010292    2.277632    0.972948     1.149358   2.088380    2.360003
 3  472   1.753349    2.138813    1.126173     1.366907   1.699985    2.078835
 4  496   2.323943    2.659027    0.817569     0.966940   1.817984    2.180067,
 (33, 7))

## Summary table

In [4]:
summary = pd.DataFrame({
    'model': ['naive', 'seasonal_naive', 'arima(2,1,2)'],
    'MAE': [res['naive_mae'].mean(), res['snaive_mae'].mean(), res['arima_mae'].mean()],
    'RMSE': [res['naive_rmse'].mean(), res['snaive_rmse'].mean(), res['arima_rmse'].mean()],
}).sort_values('RMSE')
summary

,model,MAE,RMSE
1,seasonal_naive,0.974678,1.216455
0,naive,1.504584,1.799557
2,"arima(2,1,2)",1.580653,1.888399
